# 5. Downstream Analyses of Factor Loadings

Here we use the loadings obtained with Tensor-cell2cell for each element in their respective dimensions (contexts, ligand-receptor pairs, sender cells, receiver cells). These loadings represent the importance of each element in a given factor or latent pattern. We can use this information to gain more insights about the biological processes underlying each of the communication patterns.

In [ ]:
import cell2cell as c2c

import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Directories

In [ ]:
#working env
import os
#os.chdir(r'\\isdsyn1-1.isd.med.uni-muenchen.de\BD-Bernhagen\Simon Ebert\snRNAseq_Python_env_normalMT')
os.chdir(r'W://Simon Ebert\snRNAseq_Python_env_normalMT')
os.listdir()

In [ ]:
data_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/quickstart/'
output_folder = '//isdsyn1-1.isd.med.uni-muenchen.de/BD-Bernhagen/Simon Ebert/snRNAseq_Python_env_normalMT/cell2cell_tensor/output/'
c2c.io.directories.create_directory(data_folder)
c2c.io.directories.create_directory(output_folder)

## Load Data

**Load the loadings obtained from the tensor factorization**

In [ ]:
factors = c2c.io.load_tensor_factors(output_folder + '/Loadings.xlsx')

**Dictionary with the condition of each sample/context**

This was generated when preprocessing the data in the [Notebook to Generate the 4D-Tensor](./03-Generate-Tensor.ipynb).

In [ ]:
from collections import defaultdict

element_dict = defaultdict(lambda: 'Unknown')

In [ ]:
context_dict = element_dict.copy()

context_dict.update({'HFD' : 'HFD+vehicle',
                     'HFD_L1' : 'HFD+msR4M-L1',
              
                    })

In [ ]:
from matplotlib.pyplot import rc_context
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 12
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

## Downstream Analyses

### Boxplots to statistically compare group of samples/contexts

The statistical test and the multiple test correction can be changed by modifying the parameters

```statistical_test='t-test_ind'```

```pval_correction='benjamini-hochberg'```

In [ ]:
groups_order = ["HFD+vehicle", "HFD+msR4M-L1"]
fig_filename = output_folder + '/BALF-Severity-Boxplots.pdf'
with rc_context({"figure.figsize": (4, 3)}):
    _ = c2c.plotting.context_boxplot(context_loadings=factors['Contexts'],
                                    metadict=context_dict,
                                    nrows=3,
                                    figsize=(10, 10),
                                    group_order=groups_order,
                                    statistical_test='t-test_ind',
                                    pval_correction='fdr_bh',
                                    cmap='plasma',
                                    verbose=False, 
                                    filename=fig_filename
                                    )

Here, we see that the communication patterns (or context loadings) identified statistically significant patterns when comparing each pair of COVID-19 severities.

These factors thus represent differences in the ligand-receptor interactions as well as cell types participating in cell-cell communication among the different COVID-19 severities.

### Correlation between Context Loadings & COVID-19 Severity

We first define the ranking of the severity, where higher values represent the worst severity.

In [ ]:
def severity_rank(x):
    if x == 'HFD+vehicle':
        ranking = 1
    elif x == 'HFD+msR4M-L1':
        ranking = 2
    return ranking

We generate a dataframe with the same indexes as the dataframe containing the loadings, then assign those rankings here.

In [ ]:
sev_rank = pd.DataFrame(index=factors['Contexts'].index)
sev_rank['Severity-Ranking'] = [severity_rank(context_dict[idx]) for idx in factors['Contexts'].index]
sev_rank.head()

Number of factors from the tensor decomposition

In [ ]:
nfactors = factors['Contexts'].shape[1]

**Spearman correlation**

In [ ]:
import scipy
for i in range(1, nfactors+1):
    factor = 'Factor {}'.format(i)
    print(factor, scipy.stats.spearmanr(factors['Contexts'][factor],
                                sev_rank['Severity-Ranking']))

### Heatmaps

**Cluster samples/contexts by their importance across factors**

Here we provide a example on clustering samples (colored by COVID-19 severity). If you plan adding colors for additional data and patient's properties. A detailed guide can be found here: https://www.chrisremmel.com/post/seaborn-color-labels/

Once you have a dataframe or a dictionary for coloring patients/samples, the parameter ```col_colors``` should be changed. Make sure to use the same sample labels as in ```factors['Contexts'].index```

In [ ]:
# Generate color by COVID-19 severity for each sample
condition_colors = c2c.plotting.aesthetics.get_colors_from_labels(["HFD+vehicle", "HFD+msR4M-L1"],
                                                                  cmap='plasma')

# Map these colors to each sample name
color_dict = {k : condition_colors[v] for k, v in context_dict.items()}

# Generate a dataframe used as input for the clustermap
col_colors = pd.Series(color_dict)
col_colors = col_colors.to_frame()
col_colors.columns = ['Condition']

How the color dataframe looks like. If you want to show multiple properties, add new columns with the names you want to display them, and the respective colors.

In [ ]:
col_colors.head()

Now generate the clustermap of samples given their respective loadings, colored by category of COVID-19 condition.

In [ ]:
sample_cm = c2c.plotting.loading_clustermap(factors['Contexts'],
                                            use_zscore=False, # Whether standardizing the loadings across factors
                                            col_colors=col_colors, # Change this to color by other properties
                                            figsize=(6, 4),
                                            dendrogram_ratio=0.3,
                                            cbar_fontsize=12,
                                            tick_fontsize=12,
                                            filename=output_folder + 'Clustermap-Contexts.pdf'
                                           )

#Insert legend
plt.sca(sample_cm.ax_heatmap)
legend = c2c.plotting.aesthetics.generate_legend(color_dict=condition_colors,
                                                 bbox_to_anchor=(1.25, 0.9), # Position of the legend (X, Y)
                                                 title='Condition',
                                                 fontsize=12,
                                                 
                                                )

Here patients are grouped by the importance that each communication pattern (factor) has in relation to the other patients. This captures combinations of related communication patterns that explain similarities and differences at a sample-specific resolution. In this case the differences are reflected with a clear clustering by COVID-19 severity, with some interferance of a few severe samples more similar to moderate cases.

**Cluster LR pairs by their importance across factors**

To consider important LR pairs in at least one factor, set a loading_threshold. This value can be adjusted to be more stringent control the number of ligand-receptor pairs. Here we use ```loading_threshold=0.1``` to prioritize the interactions with high loadings. If we use `loading_threshold=0`, we would return all of pairs.

In [ ]:
lr_cm = c2c.plotting.loading_clustermap(factors['Ligand-Receptor Pairs'],
                                        loading_threshold=0.1, # To consider only top LRs
                                        use_zscore=False, # Whether standardizing the loadings across factors
                                        figsize=(15, 8),
                                        filename=output_folder + 'Clustermap-LRs.pdf',
                                        tick_fontsize=8,
                                        row_cluster=False # To avoid clustering of factors
                                       )

In this case, LR pairs are grouped according to how important they for one factor versus the others. This can give an idea of the molecular mechanisms that are crucial in each communication pattern.

**Cluster sender-receiver pairs for an specific factor**

Factor 3 is associated with severe COVID-19 cases (see boxplots). Thus, it would be interesting to determine the cellular interactions underlying this communication pattern. To do so, we define its Loading Product between senders and receiver cells (outer product of loadings for that factor), which represents the joint importance of the cell-cell pair:

In [ ]:
selected_factor = 'Factor 4'

In [ ]:
loading_product = c2c.analysis.tensor_downstream.get_joint_loadings(factors,
                                                                    dim1='Sender Cells',
                                                                    dim2='Receiver Cells',
                                                                    factor=selected_factor,
                                                                   )

In [ ]:
lprod_cm = c2c.plotting.loading_clustermap(loading_product.T, # Remove .T to transpose the axes
                                           use_zscore=False, # Whether standardizing the loadings across factors
                                           figsize=(4, 6),
                                           filename=output_folder + 'Clustermap-CC-Pairs.pdf',
                                           cbar_label='Loading Product',

                                          )

**Evaluation of sender-receiver pair and LR pairs used in a specific factor**

Similarly, we can check what LR pairs each cell-cell pair is using

In [ ]:
lr_cell_product = c2c.analysis.tensor_downstream.get_lr_by_cell_pairs(factors,
                                                                      lr_label='Ligand-Receptor Pairs',
                                                                      sender_label='Sender Cells',
                                                                      receiver_label='Receiver Cells',
                                                                      order_cells_by='receivers',
                                                                      factor=selected_factor,
                                                                      cci_threshold=None, # None for considering all cell-cell pairs.
                                                                      lr_threshold=0.05 # Prioritize important pairs. None for considering all of them
                                                                     )

In [ ]:
lsr_prod_cm = c2c.plotting.loading_clustermap(lr_cell_product,
                                              use_zscore=False, # Whether standardizing the loadings across factors
                                              figsize=(18, 12),
                                              filename=output_folder + 'Clustermap-LR-CC-Pairs.pdf',
                                              cbar_label='Loading Product',
                                              yticklabels=1,
                                              tick_fontsize=8,
                                             )

### Overall CCI potential: Network visualization of sender-receiver cell pairs

An interaction network can be created for each factor by using the loading product between sender and receiver cells. First we need to choose a threshold to indicate what pair of cells are interacting. To do so, we need to get all the outer products between the loadings for the sender and receiver cells dimensions across all factors.

In [ ]:
# Get all outer products as adjacency matrices, one per factor
networks = c2c.analysis.tensor_downstream.get_factor_specific_ccc_networks(factors,
                                                                           sender_label='Sender Cells',
                                                                           receiver_label='Receiver Cells',
                                                                           )

In [ ]:
# Then, flatten the adjacency matrices
network_by_factors = c2c.analysis.tensor_downstream.flatten_factor_ccc_networks(networks, orderby='receivers')

# And we can plot the distributions of the weights for each factor-specific network
_ = plt.hist(network_by_factors.values.flatten(), bins = 50)

With this distribution we can set a reasonable threshold. We can also apply some mathematical methods to select one, as for example choosing a percentile. Here we set a threshold of 0.075.

In [ ]:
threshold = 0.05

Then, we can plot all networks we are interested in:

In [ ]:

c2c.plotting.ccc_networks_plot(factors,
                               included_factors=["Factor 4"],
                               ccc_threshold=threshold, # Only important communication
                               nrows=1,
                               panel_size=(6, 6), # This changes the size of each figure panel.
                               filename=output_folder + 'Factor-Networks.pdf',
                               factor_title_size=14,
                               node_size=200,
                               edge_width=10,
                               node_label_size=12,
                               network_layout='spring', # 'spring' or 'circular'
                              

                              )

### Gini coefficients of the factor-specific cell-cell communication

The Gini coefficient quantifies the extent to which sender-receiver cell pairs differ within a factor according to their importance as measured by their loadings. Here it is use to measure the imbalance of communication for a given factor. In other words, it identifies whether a few cells are key for an identified pattern (low Gini coefficient) or whether the pattern corresponds to a general process that involves most of the cell types (high Gini coefficient). Gini coefficients range between [0, 1]

In [ ]:
c2c.analysis.tensor_downstream.compute_gini_coefficients(factors,
                                                         sender_label='Sender Cells',
                                                         receiver_label='Receiver Cells'
                                                        ).sort_values('Gini')

We observe that the lowest Gini coefficient was obtained by factor 1, which seems to have more balanced cell loadings. Senders and/or receivers are more similar or homogeneous.